### 성공한 카페의 기준은 무엇일까?

- 상권 추정매출을 카페 단위로 분배한다(별점 및 리뷰수를 활용한다)
- 카페 밀집도를 고려해 경쟁 강도를 보정한다
- 상권별 유동인구 지표를 보조로 활용한다

#### 네이버의 카페 평점/리뷰 수/리뷰일자를 크롤링해본다

In [2]:
# Stage 1: 라이브러리 불러오기 및 데이터 로드
import os
import pandas as pd
import numpy as np

proj_dir = r"c:\Users\USER\Desktop\bootcamp\Final-Project-2_Team1"
os.chdir(proj_dir)

In [3]:
# Stage 2: Data Integration via existing mapping (no spatial join)

import pandas as pd

# 1) 리뷰·별점 포함된 카페 원본 불러오기
cafes = pd.read_csv(
    "merged_seoul_cafes.csv",
    encoding="utf-8-sig",
    dtype={"상가업소번호": str},
    low_memory=False
)

# 2) 카페→상권 매핑 파일 로드
mapping = pd.read_csv(
    "data/카페_상권_매핑_데이터.csv",
    encoding="cp949",
    dtype={"상가업소번호": str},
    low_memory=False
)

# 3) 매핑 컬럼명 통일 및 정수형 변환
mapping = mapping.rename(columns={"상권코드": "area_code"})
mapping["area_code"] = pd.to_numeric(
    mapping["TRDAR_CD"], errors="coerce"
).astype("Int64")

# 4) Inner merge
cafes_area = cafes.merge(
    mapping[["상가업소번호", "area_code"]],
    on="상가업소번호",
    how="inner"
)
df_sales = pd.read_csv("data/서울시 상권분석서비스(추정매출-상권)_2024년.csv", encoding='utf-8')
df_sales_filtered = df_sales[df_sales["서비스_업종_코드_명"].isin(["카페", "커피-음료"])]

# 5) 매핑용 Series 생성
sales_map = (
    df_sales_filtered
    .groupby("상권_코드")["당월_매출_금액"]
    .sum()
)
road_map     = pd.read_csv("data/서울시 상권분석서비스(길단위인구-상권).csv",
                          encoding='cp949', low_memory=False) \
                  .groupby("상권_코드")["총_유동인구_수"].sum()
resident_map = pd.read_csv("data/서울시 상권분석서비스(상주인구-상권).csv",
                          encoding='cp949', low_memory=False) \
                  .groupby("상권_코드")["총_상주인구_수"].sum()
work_map     = pd.read_csv("data/서울시 상권분석서비스(직장인구-상권).csv",
                          encoding='cp949', low_memory=False) \
                  .groupby("상권_코드")["총_직장_인구_수"].sum()
attr_map     = pd.read_csv("data/서울시 상권분석서비스(집객시설-상권).csv",
                          encoding='cp949', low_memory=False) \
                  .groupby("상권_코드")["집객시설_수"].sum()

# 6) map 으로 피처 결합
cafes_area["area_sales"]       = cafes_area["area_code"].map(sales_map)
cafes_area["road_pop"]         = cafes_area["area_code"].map(road_map)
cafes_area["resident_pop"]     = cafes_area["area_code"].map(resident_map)
cafes_area["work_pop"]         = cafes_area["area_code"].map(work_map)
cafes_area["attraction_count"] = cafes_area["area_code"].map(attr_map)

# 7) 결과 확인
print("▶ cafes_area with merged features (only mapped cafes):")
display(cafes_area.head())

▶ cafes_area with merged features (only mapped cafes):


,상가업소번호,상호명,kakao_score,kakao_reviewCount,naver_score,naver_reviewCount,total_reviewCount,avg_score,area_code,area_sales,road_pop,resident_pop,work_pop,attraction_count
0,MA010120220806361295,000케이크바,NaN,0,NaN,0,0,NaN,3120245,2.643649e+09,27181939,55926.0,59643.0,1394.0
1,MA0101202406A0096622,0122커피바,5.0,10,NaN,32,42,5.0,3110503,1.013111e+09,42114947,106251.0,19528.0,527.0
2,MA0101202409A0196345,0125커피바,5.0,12,NaN,14,26,5.0,3110461,7.046844e+07,6906946,34698.0,1508.0,119.0
3,MA0101202306A0062257,025베이커리,4.2,103,NaN,82,185,4.2,3110811,NaN,11440680,35341.0,4265.0,102.0
4,MA0101202406A0132263,09커피,3.6,23,NaN,6,29,3.6,3110646,5.422161e+08,45770797,180915.0,39507.0,425.0


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1) cafes_area 확인
if 'cafes_area' not in globals():
    raise RuntimeError("cafes_area가 정의되지 않았습니다. 먼저 Stage 2를 실행해주세요.")

# 2) 타깃과 피처 선택
df = cafes_area.copy()

# 타깃: 상권 추정 매출 로그 변환
df['area_sales_log'] = np.log1p(df['area_sales'].fillna(0))

# 피처: 유동인구, 상주인구, 직장인구, 집객시설 수 로그 변환
feature_cols = ['road_pop', 'resident_pop', 'work_pop', 'attraction_count']
for col in feature_cols:
    df[f'{col}_log'] = np.log1p(df[col].fillna(0))

# 3) 결측 처리
log_cols = [f'{c}_log' for c in feature_cols] + ['area_sales_log']
df[log_cols] = df[log_cols].fillna(0)

# 4) 모델링용 데이터 준비
X = df[[f'{c}_log' for c in feature_cols]]
y = df['area_sales_log']

# 5) Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 6) 스케일링
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# 7) 결과 확인
print("▶ X_train shape:", X_train.shape)
print("▶ X_test  shape:", X_test.shape)
print("▶ y_train shape:", y_train.shape)
print("▶ y_test  shape:", y_test.shape)

▶ X_train shape: (14618, 4)
▶ X_test  shape: (3655, 4)
▶ y_train shape: (14618,)
▶ y_test  shape: (3655,)


In [5]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1) Linear Regression
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

# MSE → RMSE
mse_lr  = mean_squared_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)
mae_lr  = mean_absolute_error(y_test, y_pred_lr)
r2_lr   = r2_score(y_test, y_pred_lr)

# 2) Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)

mse_rf  = mean_squared_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
mae_rf  = mean_absolute_error(y_test, y_pred_rf)
r2_rf   = r2_score(y_test, y_pred_rf)

# 3) 결과 정리
import pandas as pd
results = pd.DataFrame({
    "Model": ["LinearRegression", "RandomForest"],
    "RMSE":  [rmse_lr,  rmse_rf],
    "MAE":   [mae_lr,   mae_rf],
    "R2":    [r2_lr,    r2_rf]
})

# 4) Display results
print("▶ Model Performance (log-transformed target)")
results

▶ Model Performance (log-transformed target)


,Model,RMSE,MAE,R2
0,LinearRegression,4.332406,2.357655,0.321019
1,RandomForest,1.655315,0.286122,0.900880


In [6]:
# Stage 5: 상권 매출을 카페별로 분배하고 저장하기
import numpy as np

# 예를 들어, Stage3와 동일하게
feature_cols = ['road_pop', 'resident_pop', 'work_pop', 'attraction_count']
X_full = cafes_area.copy()
for col in feature_cols:
    X_full[f'{col}_log'] = np.log1p(X_full[col].fillna(0))
X_scaled = scaler.transform(X_full[[f'{c}_log' for c in feature_cols]])

# 1) 카페별 "가중치"로 사용할 예측값(pred_score) 생성
#    여기서는 로그 스케일 예측값을 그대로 weight로 사용
X_full['pred_score'] = rf.predict(X_scaled)

# 2) 상권별 합계로 정규화
#    (상권별 스코어 총합)
sum_scores = X_full.groupby('area_code')['pred_score'].transform('sum')
#    카페별 분배 매출
X_full['cafes_sales'] = X_full['pred_score'] / sum_scores * X_full['area_sales']

# 3) 결과 확인
print("▶ 분배된 카페별 추정매출 예시")
display(
    X_full[
        ['상가업소번호','상호명','area_code','cafes_sales']
    ].head()
)

# 4) CSV로 저장
X_full[
    ['상가업소번호','상호명','area_code','cafes_sales']
].to_csv("cafes_estimated_sales.csv", index=False, encoding='utf-8-sig')
print("✅ cafes_estimated_sales.csv 파일이 생성되었습니다.")

▶ 분배된 카페별 추정매출 예시


,상가업소번호,상호명,area_code,cafes_sales
0,MA010120220806361295,000케이크바,3120245,1.201659e+08
1,MA0101202406A0096622,0122커피바,3110503,4.221296e+07
2,MA0101202409A0196345,0125커피바,3110461,1.006692e+07
3,MA0101202306A0062257,025베이커리,3110811,NaN
4,MA0101202406A0132263,09커피,3110646,3.012311e+07


✅ cafes_estimated_sales.csv 파일이 생성되었습니다.


In [7]:
# 코드 실행 상태가 초기화되어 다시 필요한 패키지 및 파일 로드부터 진행
import pandas as pd

# 업로드된 결과 파일 재로드
df_result = pd.read_csv("cafes_estimated_sales.csv", encoding="utf-8-sig")

# cafes_sales가 NaN인 샘플 확인
nan_sales = df_result[df_result['cafes_sales'].isna()]

# 몇 개가 NaN인지, 예시 몇 개만 출력
nan_count = nan_sales.shape[0]
nan_sales_preview = nan_sales.head()

nan_count, nan_sales_preview

(1036,
                   상가업소번호        상호명  area_code  cafes_sales
 3   MA0101202306A0062257    025베이커리    3110811          NaN
 21  MA010120220814311034     120겹파이    3110540          NaN
 27  MA0101202406A0501175      15아지트    3111018          NaN
 36  MA010120220800991869        207    3110874          NaN
 49  MA010120220812810666  24시무인셀프카페    3110619          NaN)

In [8]:
import pandas as pd

# 1. 파일 경로 설정 (경로는 환경에 맞게 수정하세요)
sales_path = "cafes_estimated_sales.csv"
store_path = "data/소상공인시장진흥공단_상가(상권)정보_서울_202412.csv"

# 2. 추정매출 결과 데이터 로드
df_sales_result = pd.read_csv(sales_path, encoding="utf-8-sig")

# 3. 매출이 NaN인 카페만 필터
missing_cafes = df_sales_result[df_sales_result["cafes_sales"].isna()]

# 4. 상권 업종정보 데이터 로드
df_store = pd.read_csv(store_path, encoding="utf-8")

# 5. 업종 정보 병합 (상가업소번호 기준)
merged = pd.merge(
    missing_cafes,
    df_store,
    on="상가업소번호",
    how="left"
)

# 누락된 카페들의 실제 업종명 분포 확인
print("누락된 카페들의 업종명(중분류) 분포:")
print(merged["상권업종중분류명"].value_counts())

print("누락된 카페들의 업종명(소분류) 분포:")
print(merged["상권업종소분류명"].value_counts())

# 예시로 일부 샘플 출력
print("누락된 카페 업종 예시:")
print(merged[["상가업소번호", "상호명_x", "상권업종중분류명", "상권업종소분류명"]].head(10))

누락된 카페들의 업종명(중분류) 분포:
상권업종중분류명
비알코올     1036
Name: count, dtype: int64
누락된 카페들의 업종명(소분류) 분포:
상권업종소분류명
카페    1036
Name: count, dtype: int64
누락된 카페 업종 예시:
                 상가업소번호      상호명_x 상권업종중분류명 상권업종소분류명
0  MA0101202306A0062257    025베이커리    비알코올        카페
1  MA010120220814311034     120겹파이    비알코올        카페
2  MA0101202406A0501175      15아지트    비알코올        카페
3  MA010120220800991869        207    비알코올        카페
4  MA010120220812810666  24시무인셀프카페    비알코올        카페
5  MA0106202201A0126938  24시무인셀프카페    비알코올        카페
6  MA010120220805490968  24시무인셀프카페    비알코올        카페
7  MA0101202303A0098422       26카페    비알코올        카페
8  MA0106202403A0141067      328커피    비알코올        카페
9  MA010120220812650265       55커피    비알코올        카페


In [9]:
print("📉 area_sales NaN 개수:", cafes_area["area_sales"].isna().sum())
X_full['sum_scores'] = X_full.groupby("area_code")['pred_score'].transform('sum')
print("📉 sum_scores == 0 인 상권 수:", (X_full['sum_scores'] == 0).sum())

📉 area_sales NaN 개수: 1036
📉 sum_scores == 0 인 상권 수: 316


In [10]:
# 매출 데이터가 없는 상권 코드 리스트
missing_sales_areas = cafes_area[cafes_area["area_sales"].isna()]["area_code"].dropna().unique()

# 몇 개인지 확인
print("📉 매출 데이터가 아예 없는 상권 수:", len(missing_sales_areas))

# 예시 10개 출력
print("예시 상권 코드:", missing_sales_areas[:10])

pd.Series(missing_sales_areas, name="상권_코드").to_csv("매출_없는_상권_코드목록.csv", index=False, encoding="utf-8-sig")
print("✅ '매출_없는_상권_코드목록.csv' 저장 완료")

📉 매출 데이터가 아예 없는 상권 수: 401
예시 상권 코드: <IntegerArray>
[3110811, 3110540, 3111018, 3110874, 3110619, 3110851, 3110356, 3110741,
 3130283, 3130047]
Length: 10, dtype: Int64
✅ '매출_없는_상권_코드목록.csv' 저장 완료


In [12]:
missing_cafes = X_full[X_full["cafes_sales"].isna()]
print("cafes_sales NaN 개수:", missing_cafes.shape[0])

cafes_sales NaN 개수: 1036
